# Fine-tune Llama 3.2 with QLoRA on an Academic Q&A dataset
Run on Google Colab (free T4 GPU) or any CUDA machine. Steps: dataset -> QLoRA -> save adapter -> test unseen questions -> compare base vs tuned -> Ollama.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets pandas sentencepiece

In [ ]:
import json, os, torch, random, pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime > Change runtime type > T4 GPU)"
random.seed(42); torch.manual_seed(42)

# Ungated mirror of meta-llama/Llama-3.2-1B-Instruct (no HF login needed)
BASE_MODEL  = "unsloth/Llama-3.2-1B-Instruct"
ADAPTER_DIR = "llama32_academic_lora"
print(torch.cuda.get_device_name(0))

## 1. Dataset (academic questions + answers in a consistent tutor style)

In [ ]:
def qa(q, d, e, x):
    return {"question": q, "answer": f"Definition: {d}\nExplanation: {e}\nExample: {x}"}

train_data = [
 qa("What is photosynthesis?", "Photosynthesis is the process by which plants, algae and some bacteria convert light energy into chemical energy stored in glucose.", "Chlorophyll absorbs sunlight; water and carbon dioxide are converted into glucose and oxygen.", "A leaf using sunlight to make sugar while releasing oxygen."),
 qa("What is Newton's second law of motion?", "The acceleration of an object is directly proportional to the net force on it and inversely proportional to its mass (F = ma).", "A larger force produces a larger acceleration, while a heavier object accelerates less for the same force.", "Pushing an empty shopping cart accelerates it more than pushing a full one."),
 qa("What is a prime number?", "A prime number is a natural number greater than 1 that has exactly two divisors: 1 and itself.", "Numbers such as 2, 3, 5 and 7 cannot be divided evenly by any other number.", "13 is prime because only 1 and 13 divide it evenly."),
 qa("What is the Pythagorean theorem?", "In a right triangle, the square of the hypotenuse equals the sum of the squares of the other two sides (a^2 + b^2 = c^2).", "It relates the three side lengths and is used to compute unknown distances.", "A triangle with sides 3 and 4 has hypotenuse 5."),
 qa("What is supply and demand?", "Supply and demand is an economic model describing how price is determined by the quantity sellers offer and the quantity buyers want.", "When demand rises and supply stays fixed, prices increase; when supply rises, prices tend to fall.", "Umbrella prices rise during a sudden rainy season."),
 qa("What is inflation?", "Inflation is the sustained increase in the general price level of goods and services over time.", "As prices rise, each unit of currency buys fewer goods, reducing purchasing power.", "If a loaf of bread rises from 20 to 22 rupees in a year, that reflects 10% inflation."),
 qa("What is DNA?", "DNA (deoxyribonucleic acid) is the molecule that carries genetic instructions for the growth and functioning of living organisms.", "It is a double helix built from four bases (A, T, C, G) whose sequence encodes genes.", "The DNA sequence of a gene determines the protein a cell produces."),
 qa("What is mitosis?", "Mitosis is a type of cell division that produces two genetically identical daughter cells.", "It has phases: prophase, metaphase, anaphase and telophase, followed by cytokinesis.", "Skin cells divide by mitosis to heal a cut."),
 qa("What is an algorithm?", "An algorithm is a finite, step-by-step procedure for solving a problem or performing a computation.", "It takes input, follows unambiguous steps and produces output, and is judged by correctness and efficiency.", "Binary search finds a name in a sorted list by repeatedly halving the search range."),
 qa("What is machine learning?", "Machine learning is a field of artificial intelligence in which systems learn patterns from data instead of being explicitly programmed.", "Models are trained on examples to minimize an error function and then make predictions on new data.", "A spam filter learns from labeled emails to classify new messages."),
 qa("What is a neural network?", "A neural network is a computational model made of layers of connected nodes (neurons) inspired by the brain.", "Each connection has a weight; training adjusts weights with backpropagation to reduce prediction error.", "An image classifier uses layers of neurons to recognize handwritten digits."),
 qa("What is the water cycle?", "The water cycle is the continuous movement of water between the oceans, atmosphere and land.", "Water evaporates, condenses into clouds, falls as precipitation and flows back through rivers and groundwater.", "Rain that falls on a hill eventually flows into a river and returns to the sea."),
 qa("What is gravity?", "Gravity is the force of attraction between objects with mass.", "Its strength grows with mass and weakens with the square of the distance between objects.", "Gravity keeps the Moon in orbit around the Earth."),
 qa("What is the French Revolution?", "The French Revolution (1789-1799) was a period of radical political and social change in France that ended the absolute monarchy.", "Economic hardship and Enlightenment ideas led to the fall of the Bastille and the rise of the republic.", "The storming of the Bastille in 1789 became a symbol of the revolution."),
 qa("What is democracy?", "Democracy is a system of government in which power is held by the people, directly or through elected representatives.", "Citizens vote in free elections, and rights and the rule of law limit government power.", "India holds general elections where citizens choose members of parliament."),
 qa("What is an atom?", "An atom is the smallest unit of a chemical element that keeps the element's properties.", "It has a nucleus of protons and neutrons surrounded by orbiting electrons.", "A carbon atom has 6 protons and 6 electrons."),
 qa("What is the periodic table?", "The periodic table is a tabular arrangement of chemical elements ordered by atomic number.", "Elements in the same column share similar chemical properties because of similar electron configurations.", "Sodium and potassium are in the same group and both react vigorously with water."),
 qa("What is an acid?", "An acid is a substance that donates hydrogen ions (H+) in water and has a pH below 7.", "Acids taste sour, turn blue litmus red and react with bases to form salts and water.", "Vinegar contains acetic acid."),
 qa("What is a derivative in calculus?", "A derivative measures the instantaneous rate of change of a function with respect to its variable.", "Geometrically it is the slope of the tangent line to the curve at a point.", "The derivative of x^2 is 2x."),
 qa("What is an integral in calculus?", "An integral computes the accumulated quantity under a curve, such as area.", "It is the reverse of differentiation, as stated by the Fundamental Theorem of Calculus.", "The integral of 2x is x^2 + C."),
 qa("What is probability?", "Probability is a number between 0 and 1 that measures how likely an event is to occur.", "It is computed as favorable outcomes divided by total equally likely outcomes.", "The probability of rolling a 4 on a fair die is 1/6."),
 qa("What is the mean of a data set?", "The mean is the sum of all values divided by the number of values.", "It measures the central tendency but is sensitive to extreme values.", "The mean of 2, 4 and 9 is 5."),
 qa("What is a database?", "A database is an organized collection of structured data stored and accessed electronically.", "A database management system lets users insert, query, update and secure the data.", "A university stores student records in a relational database."),
 qa("What is object-oriented programming?", "Object-oriented programming (OOP) is a paradigm that organizes software around objects that combine data and behavior.", "Its core principles are encapsulation, inheritance, polymorphism and abstraction.", "A Car class with speed and a drive() method can be inherited by ElectricCar."),
 qa("What is the greenhouse effect?", "The greenhouse effect is the warming of Earth as atmospheric gases trap heat radiated from the surface.", "Gases such as carbon dioxide and methane absorb infrared radiation and re-emit it downward.", "Rising CO2 from burning fossil fuels strengthens the effect and raises global temperatures."),
 qa("What is natural selection?", "Natural selection is the process by which organisms with advantageous traits survive and reproduce more than others.", "Over generations, favorable heritable traits become more common in a population.", "Darwin's finches developed different beak shapes suited to available food."),
 qa("What is a metaphor?", "A metaphor is a figure of speech that describes one thing as if it were another to suggest a similarity.", "Unlike a simile, it does not use 'like' or 'as'.", "\"Time is a thief\" implies that time steals moments from us."),
 qa("What is the Industrial Revolution?", "The Industrial Revolution was the transition from hand production to machine manufacturing beginning in Britain in the 18th century.", "Steam power, factories and new transport transformed economies and moved people to cities.", "The spinning jenny greatly increased textile output."),
 qa("What is GDP?", "Gross Domestic Product (GDP) is the total market value of all final goods and services produced in a country in a given period.", "Economists use it to measure the size and growth of an economy.", "If a country's GDP grows from 100 to 105 billion, it grew by 5%."),
 qa("What is osmosis?", "Osmosis is the movement of water across a semipermeable membrane from a region of low solute concentration to high solute concentration.", "It continues until concentrations equalize or pressure balances the flow.", "A raisin placed in water swells as water moves into it."),
 qa("What is Ohm's law?", "Ohm's law states that the current through a conductor is proportional to the voltage across it (V = IR).", "For a fixed resistance, doubling the voltage doubles the current.", "A 10 V battery across a 5 ohm resistor drives a 2 A current."),
 qa("What is recursion in computer science?", "Recursion is a technique in which a function solves a problem by calling itself on smaller subproblems.", "It needs a base case to stop and a recursive case that moves toward it.", "Factorial: n! = n * (n-1)!, with 0! = 1."),
 qa("What is cellular respiration?", "Cellular respiration is the process by which cells break down glucose to release usable energy as ATP.", "It includes glycolysis, the Krebs cycle and the electron transport chain, mostly in the mitochondria.", "Muscle cells use it to power a run."),
 qa("What is the Doppler effect?", "The Doppler effect is the change in observed frequency of a wave when the source and observer move relative to each other.", "Approaching sources compress waves (higher pitch); receding sources stretch them (lower pitch).", "An ambulance siren sounds higher as it approaches and lower as it leaves."),
]

# Unseen questions (NOT in the training set)
test_questions = [
 "What is entropy in thermodynamics?",
 "What is a linear regression?",
 "What is the Renaissance?",
 "What is meiosis?",
 "What is opportunity cost?",
 "What is a binary tree in computer science?",
]
train_titles = {d["question"] for d in train_data}
assert not (set(test_questions) & train_titles)

os.makedirs("data", exist_ok=True)
with open("data/academic_qa_train.jsonl", "w") as f:
    for d in train_data:
        f.write(json.dumps({"messages": [
            {"role": "system", "content": "You are an academic tutor. Answer with Definition, Explanation and Example."},
            {"role": "user", "content": d["question"]},
            {"role": "assistant", "content": d["answer"]}]}) + "\n")
print(len(train_data), "training examples saved to data/academic_qa_train.jsonl")

## 2. Load the base model in 4-bit (QLoRA)

In [ ]:
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=compute_dtype)
model.config.use_cache = False
print("Loaded. GPU memory (GB):", round(torch.cuda.memory_allocated()/1e9, 2))

## 3. Tokenize (loss only on the assistant answer)

In [ ]:
SYSTEM = "You are an academic tutor. Answer with Definition, Explanation and Example."
MAX_LEN = 512

def build(example):
    msgs = example["messages"]
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    answer = msgs[-1]["content"] + "<|eot_id|>"
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100]*len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "labels": labels}

raw = Dataset.from_json("data/academic_qa_train.jsonl")
ds = raw.map(build, remove_columns=raw.column_names)

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), m), tokenizer.pad_token_id)
    lab = torch.full((len(batch), m), -100)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

print(ds)

## 4. Answers from the ORIGINAL model (adapter not yet attached)

In [ ]:
def ask(m, question, max_new_tokens=220):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(m.device)
    m.config.use_cache = True
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")])
    m.config.use_cache = False
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

base_answers = {q: ask(model, q) for q in test_questions}
for q, a in base_answers.items():
    print("Q:", q, "\nA:", a, "\n" + "-"*70)

## 5. Attach LoRA adapters and train

In [ ]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="lora_checkpoints", num_train_epochs=6, per_device_train_batch_size=4,
    gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    logging_steps=2, save_strategy="no", optim="paged_adamw_8bit",
    bf16=use_bf16, fp16=not use_bf16, report_to="none", remove_unused_columns=False, seed=42)

trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
trainer.train()

## 6. Save the LoRA adapter

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(os.listdir(ADAPTER_DIR))
!du -sh {ADAPTER_DIR}

## 7. Test the customized model on unseen questions and compare with the original

In [ ]:
tuned_answers = {q: ask(model, q) for q in test_questions}

# Same loaded weights, adapter switched off -> exact original-model behaviour (sanity check)
with model.disable_adapter():
    check = ask(model, test_questions[0])
print("Adapter-off matches earlier base answer:", check == base_answers[test_questions[0]], "\n")

for q in test_questions:
    print("="*80); print("QUESTION:", q)
    print("\n--- ORIGINAL ---\n" + base_answers[q])
    print("\n--- CUSTOMIZED (LoRA) ---\n" + tuned_answers[q])

In [ ]:
def follows_format(t): return all(k in t for k in ("Definition:", "Explanation:", "Example:"))
df = pd.DataFrame({
    "question": test_questions,
    "orig_words": [len(base_answers[q].split()) for q in test_questions],
    "tuned_words": [len(tuned_answers[q].split()) for q in test_questions],
    "orig_follows_format": [follows_format(base_answers[q]) for q in test_questions],
    "tuned_follows_format": [follows_format(tuned_answers[q]) for q in test_questions],
})
display(df)
print("Format adherence -> original: %d/%d | customized: %d/%d" % (
    df.orig_follows_format.sum(), len(df), df.tuned_follows_format.sum(), len(df)))
df.to_csv("comparison_results.csv", index=False)

## 8. Use with local Ollama
Option A (adapter): Ollama applies the LoRA adapter on top of the matching base model.
Option B (merged GGUF): merge the adapter into the full model and convert to GGUF.

Ollama must run on the machine where you want to chat (on Colab it is installed below; locally, copy `llama32_academic_lora/` and the Modelfile to your PC and run the `ollama create` command).

In [ ]:
# Modelfile for Option A: base = ollama's llama3.2:1b (must match the base the adapter was trained on)
modelfile = f"""FROM llama3.2:1b
ADAPTER ./{ADAPTER_DIR}
SYSTEM \"\"\"{SYSTEM}\"\"\"
PARAMETER temperature 0.2
"""
open("Modelfile", "w").write(modelfile)
print(modelfile)

In [ ]:
# Install and start Ollama (Linux/Colab). Skip if already installed.
import shutil, subprocess, time
if shutil.which("ollama") is None:
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull llama3.2:1b

In [ ]:
!ollama create academic-llama -f Modelfile
!ollama list

In [ ]:
import requests
def ollama_ask(model_name, question):
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": model_name, "stream": False, "options": {"temperature": 0},
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]}, timeout=300)
    return r.json()["message"]["content"]

q = "What is a hash table in computer science?"   # new question, not in training
print("### llama3.2:1b (original, Ollama)\n", ollama_ask("llama3.2:1b", q))
print("\n### academic-llama (customized, Ollama)\n", ollama_ask("academic-llama", q))

### Option B (optional): merge + GGUF for Ollama

In [ ]:
# Merge LoRA into a full fp16 model, then convert to GGUF with llama.cpp
del model, trainer
import gc; gc.collect(); torch.cuda.empty_cache()

base16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base16, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained("llama32_academic_merged", safe_serialization=True)
tokenizer.save_pretrained("llama32_academic_merged")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp 2>/dev/null || git clone -q --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf numpy sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py llama32_academic_merged --outfile academic-llama-f16.gguf --outtype f16

open("Modelfile.merged", "w").write(f"""FROM ./academic-llama-f16.gguf
SYSTEM \"\"\"{SYSTEM}\"\"\"
PARAMETER temperature 0.2
""")
!ollama create academic-llama-merged -f Modelfile.merged
print(ollama_ask("academic-llama-merged", "What is a hash table in computer science?"))